# Chapter 9: Applications in Symplectic Topology

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 9, printed pp. 295-368; PDF pp. 310-383. Sections: 9.1-9.7.

    ## Chapter Goal

    Hamiltonian periodic orbits, Lagrangian embeddings, nonsqueezing, symplectic four-manifolds, symplectomorphism groups, Hofer geometry, and distinguishing symplectic structures.

    The guiding question is:

    > What rigid geometric consequences can be extracted from J-holomorphic curve compactness and counting?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| capacity | monotone numerical symplectic invariant | ball vs cylinder inequality |
| nonsqueezing | embedding obstruction | pi R^2 <= pi r^2 |
| Lagrangian obstruction | disk or curve compactness argument | area/action threshold |
| ruled surface structure | foliation by curves | intersection positivity |
| Hofer geometry | energy length of Hamiltonian paths | action estimate |


## Standalone Reading Guide

    This chapter is the payoff for the analytic foundations. J-holomorphic curves turn soft-looking questions about symplectic embeddings and Hamiltonian dynamics into rigid problems constrained by area, intersection, and compactness. The nonsqueezing theorem is the cleanest visual example: a ball may fit through a cylinder only when the symplectic area seen by the complex line is large enough, no matter how much volume is available.

Other applications use the same logic in less elementary clothing. Periodic orbit results arise from studying sections or Floer-type limits. Lagrangian embedding obstructions use disks, spheres, and energy. Four-dimensional classification results exploit positivity of intersections and foliations by curves. Hofer geometry records the energy cost of Hamiltonian paths and relates dynamics to metric behavior.

The experiment focuses on capacity because it is inspectable. A grid of ball radii and cylinder radii is colored by the inequality pi R^2 <= pi r^2. The check confirms monotonicity and conformality, the two basic behaviors any symplectic capacity should satisfy. The plot is intentionally simple: it lets the learner see rigidity as a sharp boundary rather than as an abstract theorem name.

    ## Topics In This Notebook

    - periodic orbits through section counts and Hamiltonian dynamics
- Lagrangian obstructions detected by holomorphic disks or spheres
- nonsqueezing as a capacity inequality
- rational and ruled symplectic four-manifolds
- Hofer geometry and symplectomorphism-group phenomena

    ## Visualization Storyboard

    - A capacity matrix visualizes which ball-to-cylinder radius pairs pass the nonsqueezing inequality.
- A dependency graph connects compactness and positivity to several applications.
- A ledger checks monotonicity, conformality, and the sharp radius inequality.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-09'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['capacity', 'nonsqueezing', 'Lagrangian obstruction', 'ruled surface structure', 'Hofer geometry']
CONCEPT_EDGES = [('capacity', 'nonsqueezing'), ('nonsqueezing', 'Lagrangian obstruction'), ('Lagrangian obstruction', 'ruled surface structure'), ('ruled surface structure', 'Hofer geometry')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=45, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Applications in Symplectic Topology')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '295-368',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Applications in Symplectic Topology. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
R = np.linspace(0.2, 2.0, 80)
r = np.linspace(0.2, 2.0, 80)
RR, rr = np.meshgrid(R, r)
feasible = RR <= rr
capacity_ball = np.pi * RR**2
capacity_cylinder = np.pi * rr**2
margin = capacity_cylinder - capacity_ball

fig, ax = plt.subplots(figsize=(6.2, 5.5))
im = ax.imshow(feasible.astype(float), extent=[R.min(), R.max(), r.min(), r.max()], origin="lower", cmap="YlGnBu", aspect="auto")
ax.plot(R, R, color="black", linewidth=1.5, label="R = r boundary")
ax.set_xlabel("ball radius R")
ax.set_ylabel("cylinder radius r")
ax.set_title("Nonsqueezing capacity feasibility")
ax.legend()
fig.colorbar(im, ax=ax, label="embedding not obstructed by capacity")
fig_path = save_matplotlib(fig, UNIT, "figures", "nonsqueezing-capacity-region.png")
plt.close(fig)

check = {
    "min_feasible_margin": float(margin[feasible].min()),
    "max_infeasible_margin": float(margin[~feasible].max()),
    "capacity_scales_quadratically": bool(np.isclose(np.pi * (2 * 0.7) ** 2, 4 * np.pi * 0.7**2)),
    "passed": bool(margin[feasible].min() >= -1e-12 and margin[~feasible].max() < 0),
}
check_path = save_json(check, UNIT, "checks", "nonsqueezing-capacity-checks.json")
display_artifact(fig_path, width=620)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'capacity', 'computational_object': 'monotone numerical symplectic invariant', 'check': 'ball vs cylinder inequality'}, {'item': 'nonsqueezing', 'computational_object': 'embedding obstruction', 'check': 'pi R^2 <= pi r^2'}, {'item': 'Lagrangian obstruction', 'computational_object': 'disk or curve compactness argument', 'check': 'area/action threshold'}, {'item': 'ruled surface structure', 'computational_object': 'foliation by curves', 'check': 'intersection positivity'}, {'item': 'Hofer geometry', 'computational_object': 'energy length of Hamiltonian paths', 'check': 'action estimate'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Scale every radius by the same factor. The feasible region should remain the same after comparing squared radii, while capacities scale by the square of that factor.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Symplectic rigidity often appears as an area or action obstruction.
- Nonsqueezing is a capacity statement, not a volume statement.
- Four-dimensional applications rely heavily on compactness and positivity of intersections.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'nonsqueezing-capacity-region.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'nonsqueezing-capacity-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'nonsqueezing-capacity-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
